# Обучение LoRA-адаптера «Пингвин.Хост» (Colab)

Спутник урока 6.2 курса «LLM для Python-разработчиков». Запускается **на Google
Colab с GPU**: Среда выполнения → Сменить среду выполнения → **T4 GPU**.

Локально этот ноутбук запускать не нужно — он рассчитан на окружение Colab.
Понадобятся файлы `train_messages.jsonl` и `val_messages.jsonl` из `data/`
(их создаёт урок 6.2).

План: установка → данные → база в 4 битах → LoRA r=16 → обучение →
смоук-тест → сохранение адаптера и GGUF.

Ноутбук обслуживает две задачи модуля — переключатель `TASK` в ШАГЕ 2:
`"pingvin"` — стиль поддержки (уроки 6.2–6.3, 45 примеров, 8 эпох);
`"json"` — JSON-бот из капстоуна 6.4 (500+ примеров из `capstone/json_bot/`, 3 эпохи).

In [ ]:
# ШАГ 1. Установка (2-4 минуты). Точечные версии не пиним:
# unsloth сам тянет совместимые trl/peft/transformers.
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
print("unsloth установлен")

In [ ]:
# ШАГ 2. Данные: загрузите train_messages.jsonl и val_messages.jsonl
import json
from pathlib import Path

TASK = "pingvin"   # "pingvin" — стиль поддержки (уроки 6.2-6.3); "json" — JSON-бот (капстоун 6.4)
NAME = {"pingvin": "pingvin", "json": "pingvin-json"}[TASK]     # имена артефактов
EPOCHS = {"pingvin": 8, "json": 3}[TASK]   # 45 примеров -> 8 эпох (см. ШАГ 6); 500+ -> хватит 3

if not Path("train_messages.jsonl").exists():
    from google.colab import files            # noqa: в Colab модуль есть
    uploaded = files.upload()                 # выберите ОБА jsonl из data/

train_rows = [json.loads(l) for l in Path("train_messages.jsonl").read_text(encoding="utf-8").splitlines()]
val_rows = [json.loads(l) for l in Path("val_messages.jsonl").read_text(encoding="utf-8").splitlines()]
print(f"train: {len(train_rows)}, val: {len(val_rows)}")
assert train_rows[0]["messages"][0]["role"] == "user"


In [ ]:
# ШАГ 3. База в 4 битах (QLoRA из урока 6.1). Instruct, не base!
from unsloth import FastLanguageModel

MAX_SEQ = 512                                  # замерили в уроке 6.2: max ~280 токенов

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)

In [ ]:
# ШАГ 4. Навешиваем LoRA-адаптеры: ~30 млн обучаемых параметров (<1%)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# ШАГ 5. Датасет: родной шаблон чата токенизатора (никакого самодельного ChatML)
from datasets import Dataset

def to_text(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_rows).map(to_text, remove_columns=["messages"])
val_ds = Dataset.from_list(val_rows).map(to_text, remove_columns=["messages"])

print("--- первый обучающий пример (проверьте разметку и полноту!) ---")
print(train_ds[0]["text"][:500])

In [ ]:
# ШАГ 6. Тренер: Пингвин — 40 шагов (40 примеров / батч 8 * 8 эпох), JSON-бот — ~190 шагов
# (500+ примеров * 3 эпохи); loss только на ответах.
# Живой урок: 3 эпохи (15 шагов) оказалось МАЛО - merged-модель давала 1/5 якорей стиля
# даже на вопросах из train (норма LoRA-матриц B ~0.16, поправка весов ~1e-4).
# Ориентир: train loss к концу ниже ~0.5. Протечку стиля лечат данными (урок 6.4, практика 2).
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_steps=3,
        logging_steps=1,
        eval_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(               # маскируем вопросы из loss
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# ШАГ 7. Обучение (~2-4 минуты на T4). Следите за loss - разбор в уроке 6.2, раздел 8
stats = trainer.train()
print(f"\nfinal train loss: {stats.training_loss:.3f}")

In [ ]:
# ШАГ 8. Смоук-тест на отложенных вопросах (их не было в обучении)
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

STYLE_ANCHORS = ["Поддержка «Пингвин.Хост» на связи", "\U0001F427",
                 "Что происходит:", "Что делать:", "тикет остаётся открытым"]

SMOKE = {
    "pingvin": ["Что будет с моим сайтом, если я не успею вовремя оплатить тариф?",
                "У меня на почте заканчивается место, письма перестали приходить. Что делать?"],
    "json": ["Если я случайно удалю файлы, есть ли откуда восстановить?",   # ждём строку и conf >= 0.5
             "Какая столица Канады?"],                                        # ждём null / 0.0
}[TASK]

def smoke_score(answer: str) -> str:
    if TASK == "pingvin":
        return f"стиль: {sum(a in answer for a in STYLE_ANCHORS)}/5"
    try:                                        # JSON-бот: формат вместо якорей
        parsed = json.loads(answer.strip())
        ok = set(parsed) == {"answer", "confidence"}
        return f"JSON: {'ок' if ok else 'не те ключи'}, confidence={parsed.get('confidence')}"
    except (json.JSONDecodeError, AttributeError):
        return "JSON: НЕ ПАРСИТСЯ"

for question in SMOKE:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=300,
                         temperature=0.1, do_sample=True)
    answer = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"=== {question}\n{smoke_score(answer)}\n{answer[:400]}\n")


In [ ]:
# ШАГ 9. Сохраняем ОБА артефакта. GGUF-экспорт собирает llama.cpp - это ~10 минут.
model.save_pretrained(f"{NAME}-lora")            # адаптер: страховочная копия
tokenizer.save_pretrained(f"{NAME}-lora")

model.save_pretrained_gguf(f"{NAME}-gguf", tokenizer,
                           quantization_method="q4_k_m")

import shutil
shutil.make_archive(f"{NAME}-lora", "zip", f"{NAME}-lora")
print(f"готово: {NAME}-lora.zip + {NAME}-gguf/*.gguf")


In [ ]:
# ШАГ 10. Скачиваем (НЕ откладывайте: сессия Colab эфемерна!)
from pathlib import Path

from google.colab import files

# unsloth разных версий кладёт GGUF в <NAME>-gguf/ или рядом и называет его по базовой
# модели (Qwen2.5-3B-Instruct.Q4_K_M.gguf) - ищем везде и переименовываем под урок 6.3
candidates = [f for f in Path(".").rglob("*.gguf") if "llama.cpp" not in f.parts]
assert candidates, "GGUF не найден - смотрите вывод шага 9"
gguf = max(candidates, key=lambda f: f.stat().st_size)      # самый большой = модель, не mmproj
gguf = gguf.rename(gguf.with_name(f"{NAME}-q4_k_m.gguf"))
print("GGUF:", gguf, f"{gguf.stat().st_size/2**30:.2f} ГБ")
files.download(str(gguf))                        # ~2 ГБ, наберитесь терпения
files.download(f"{NAME}-lora.zip")
print("Положите оба файла в 06-fine-tuning-lora/artifacts/ - их ждут урок 6.3 и capstone/json_bot")


## Если сломалось

- **`RuntimeError: ... CUDA`/`No GPU found`** — не выбран T4: Среда выполнения →
  Сменить среду выполнения → T4 GPU, затем перезапустить ноутбук.
- **Ошибка на `SFTConfig`/`SFTTrainer` (неожиданный аргумент)** — API trl
  сдвинулся. Откройте [актуальный пример unsloth](https://docs.unsloth.ai/) для
  Qwen и сверьте имена аргументов; смысл каждого — в таблице урока 6.2.
- **OOM на шаге обучения** — уменьшите `per_device_train_batch_size` до 2
  (grad_accum поднимите до 4: эффективный батч тот же).
- **`save_pretrained_gguf` падает или бесконечно собирает llama.cpp** — скачайте
  хотя бы адаптер (`pingvin-lora.zip`). Ollama 0.33 папку safetensors для qwen2
  через `ADAPTER` не принимает (проверено: «no safetensors files found»), поэтому
  запасной путь — `convert_lora_to_gguf.py` из llama.cpp и `ADAPTER adapter.gguf`.
- **Стиль на смоук-тесте 0-2/5** — проверьте печать первого примера в ШАГЕ 5:
  видна ли разметка `<|im_start|>` и полный ответ с финалкой. Обрыв — увеличьте
  MAX_SEQ; нет разметки — датасет сохранён не тем скриптом.